In [1]:
import json

import os

d=[]
with open('D:\github4\web_driver\data.jsonl','r', encoding='utf-8') as f:
    for i in f:

        d += [json.loads(i)]

In [ ]:
i=650
print(d[i]['url'])
print(d[i]['title'])

print(d[i]['conversations'])

In [1]:

import re
import json

alinks=[]
with open('D:\github4\web_driver\data.jsonl','r',encoding='utf-8') as f:
    c=0
    for i in f:
        a=json.loads(i)
        link_pattern = re.compile(r'\[link\s+https://[^\]]+\]')

        links = link_pattern.findall(a['conversations'])
        alinks+=[j[6:-1] for j in links]

print(len(alinks))

2248


In [ ]:
import requests
path='D:\github4\web_driver\imgs'
c = 0
sc=0
for li in alinks:
    
    if '.png' in li.lower() or '.jpeg' in li.lower() or '.jpg' in li.lower() or '.bmp' in li.lower() or '.svg' in li.lower():
        image_url = li
        try:
            response = requests.get(image_url)
            if response.status_code == 200:
                pos = li.rfind('/')
                li = li[pos+1:]
                with open(f"D:\github4\web_driver\imgs\{li}", "wb") as file:
                        file.write(response.content)
                sc+=1
                print(f"Image successfully downloaded and saved as D:\github4\web_driver\imgs\{li}")
            else:
                c+=1
                print(f"Failed to download image. HTTP Status code: {response.status_code} v={image_url}")
        except:
            c+=1
            print("a failed connect")
print(f'failed {c} suce:{sc}')


In [ ]:
import os
import img2pdf

def image_to_pdf(image_path, pdf_path):
    with open(pdf_path, "wb") as f:
        f.write(img2pdf.convert(image_path))

def folder_images_to_pdfs(folder_path, output_folder):
    
    for filename in os.listdir(folder_path):
        image_path = os.path.join(folder_path, filename)
        pdf_path = f"{output_folder}\{os.path.splitext(filename)[0] + '.pdf'}"
        try:
            image_to_pdf(image_path, pdf_path)
        except:

            print(f'not good {image_path}')
        # print(f"Converted {filename} to {os.path.basename(pdf_path)}")

# 示例用法
folder_path = 'D:\github4\web_driver\imgs'
output_folder = 'D:\github4\web_driver\pdfs'
folder_images_to_pdfs(folder_path, output_folder)

In [35]:
import os
import glob

def delete_pdf_files(directory):
    # 构造搜索路径
    search_path = os.path.join(directory, '*.pdf')
    # 获取所有匹配的文件路径
    pdf_files = glob.glob(search_path)
    # 删除所有匹配的文件
    for file_path in pdf_files:
        try:
            os.remove(file_path)
            print(f"Deleted file: {file_path}")
        except Exception as e:
            print(f"Error deleting file {file_path}: {e}")
delete_pdf_files('D:\github4\web_driver\imgs')

In [ ]:
import json
k=[]
c=0
with open('D:\github4\web_driver\data_deeps_clea copy.jsonl', 'r',encoding='utf-8') as f:
    for i in f:
        k=json.loads(i)
        if not k['deep'].startswith(("**Question:**","### Question:","### Question and Answer","```json\n{\n  \"question\":","<","Network Error!","### Q&A Pairs","**Question 1","### Question 1",\
                                     "**Question by","### Problem Description","### Optimization Problem"\
                                     ,"### User's Wish List for CVX Enhancements","### Max-min QoS Fairness Optimization with MRC","**Question from","**Astrid:**"\
                                     ,"**Question a:","**Question: ","### Problem 1:")):
            print(k['deep'])
            c+=1
            
print(f"c={c}")

In [107]:
import json


def recursive_chunk(text:str, split_symbols:list[str]=[r'\n[0-9][0-9]?\.[0-9][^a-zA-Z]',],rxp=True, max_len = None)->list[str]:
    import re
    res=[text]

    if max_len:
        if len(text) < max_len:
            return res
        #apply split_symbols until chunks under max_len
        while split_symbols:
            op=split_symbols.pop(0)
            res_temp=[]
            while res:
                chunk = res.pop(0)
                if len(chunk) < max_len:
                    res_temp+=[chunk]
                else:
                    chunks=re.split(op, chunk)
                    res_temp+=chunks
            res = res_temp

        def constlen_chunk(s):
            if max_len:
                return [s[i:i+max_len] for i in range(0, len(s), max_len)]
            return s
        
        #keep them under max_len
        res_temp=[]
        while res:
            chunk = res.pop(0)
            if len(chunk) < max_len:
                res_temp+=[chunk]
            else:
                chunks=constlen_chunk(chunk)
                res_temp+=chunks
        res = res_temp


    if not max_len:
        #apply split_symbols
        while split_symbols:
            op=split_symbols.pop(0)
            res_temp=[]
            while res:
                chunk = res.pop(0)
                if rxp:
                    chunks=re.split(op, chunk)
                else:
                    chunks=chunk.split(op)
                res_temp+=chunks
            res = res_temp

    return res
def delete_small_chunks(chunks:list[str], mini_len):

    res=[]
    while chunks:
        chunk = chunks.pop(0)
        if len(chunk) >= mini_len:
            res+=[chunk]
    return res
k=[]
c=0
with open("D:\github4\web_driver\data_new.jsonl",'w',encoding="utf-8") as f2:
    with open("D:\github4\web_driver\data_new_handmade",'w',encoding="utf-8") as f_hand:
        with open('D:\github4\web_driver\data_deeps_clea copy.jsonl', 'r',encoding='utf-8') as f:
            for i in f:
                k=json.loads(i)
                if k['deep'].startswith("**Question:**",):
                    t= k['deep']
                    t=t.strip()
                    t=recursive_chunk(t,split_symbols=[r"\*\*Question:\*\*",r"\*\*Answer:\*\*"])

                    t=delete_small_chunks(t,5)


                    assert(len(t)%2==0)
                    conversations=[]
                    for j in range(int(len(t)/2)):
                        conversations+=[{"content":t[j*2].strip(), "role": "user"},{"content": t[j*2+1].strip(), "role": "assistant"}]

                    conversations={"messages":conversations}
                    f2.write(json.dumps(conversations)+'\n')

                elif k['deep'].startswith("### Question:",):
                    t= k['deep']
                    t=t.strip()
                    t=recursive_chunk(t,split_symbols=[r"### Question:",r"### Answer:"])

                    t=delete_small_chunks(t,5)


                    assert(len(t)%2==0)
                    conversations=[]
                    for j in range(int(len(t)/2)):
                        conversations+=[{"content":t[j*2].strip(), "role": "user"},{"content": t[j*2+1].strip(), "role": "assistant"}]

                    conversations={"messages":conversations}
                    f2.write(json.dumps(conversations)+'\n')
                elif k['deep'].startswith("### Question and Answer Pairs",):
                    t= k['deep'][len('### Question and Answer Pairs'):]
                    t=t.strip()
                    t0=recursive_chunk(t,split_symbols=[r"\*\*Q[0-9]:",r"\*\*\n\n\*\*A[0-9]:\*\*",r"#### Q[0-9]:",r"\*\*A[0-9]:\*\*",r"[0-9]\. \*\*Question:\*\*",\
                                                        r"\*\*Answer:\*\*",r"[0-9]\. \*\*Q:",r"\*\*\n   \*\*A:\*\*",r"\*\*\n\n\*\*A[0-9]:",r"\n\n\*\*Q[0-9]:",r'\*\*\n    \*\*A:\*\*',\
                            r"\*\*Question [0-9]:\*\*" ,     r"\*\*Answer [0-9]:\*\*" ,    r"\*\*Question:\*\*",r"\*\*Answer:\*\*",   r"\*\*\n   A: "   ,r"\*\*\n\*\*A[0-9]:"               ])

                    t=delete_small_chunks(t0.copy(),5)


                    assert(len(t)%2==0)
                    conversations=[]
                    for j in range(int(len(t)/2)):
                        conversations+=[{"content":t[j*2].strip(), "role": "user"},{"content": t[j*2+1].strip(), "role": "assistant"}]

                    conversations={"messages":conversations}
                    f2.write(json.dumps(conversations)+'\n')

                elif k['deep'].startswith("**Question 1:**",):
                    t= k['deep']
                    t=t.strip()
                    t0=recursive_chunk(t,split_symbols=[r'\*\*Question [0-9]:\*\*',r'\*\*Answer [0-9]:\*\*',])

                    t=delete_small_chunks(t0.copy(),5)


                    assert(len(t)%2==0)
                    conversations=[]
                    for j in range(int(len(t)/2)):
                        conversations+=[{"content":t[j*2].strip(), "role": "user"},{"content": t[j*2+1].strip(), "role": "assistant"}]

                    conversations={"messages":conversations}
                    f2.write(json.dumps(conversations)+'\n')
                elif k['deep'].startswith("### Q&A Pairs",): 
                    t= k['deep'][len("### Q&A Pairs"):]
                    t=t.strip()
                    t0=recursive_chunk(t,split_symbols=[r'[0-9]\. \*\*Q:',r'\*\*\n   \*\*A:\*\*'],)

                    t=delete_small_chunks(t0.copy(),5)

                    assert(len(t)%2==0)
                    conversations=[]
                    for j in range(int(len(t)/2)):
                        conversations+=[{"content":t[j*2].strip(), "role": "user"},{"content": t[j*2+1].strip(), "role": "assistant"}]

                    conversations={"messages":conversations}
                    f2.write(json.dumps(conversations)+'\n')
                elif k['deep'].startswith("### Question 1:",): 
                    t= k['deep']
                    t=t.strip()
                    t0=recursive_chunk(t,split_symbols=[r'### Question [0-9]:',r'### Answer [0-9]:',r"\*\*Q:\*\*",r"\*\*A:\*\*"\
                                                        ,r'\*\*Question:\*\*',r'\*\*Answer:\*\*'],)

                    t=delete_small_chunks(t0.copy(),5)
                    assert(len(t)%2==0)
                    conversations=[]
                    for j in range(int(len(t)/2)):
                        conversations+=[{"content":t[j*2].strip(), "role": "user"},{"content": t[j*2+1].strip(), "role": "assistant"}]

                    conversations={"messages":conversations}
                    f2.write(json.dumps(conversations)+'\n')
                else:
                    f_hand.write(k['deep']+'\n--------------new--post--line--------------------------------\n')

In [113]:
with open("D:\github4\web_driver\data_new_2.jsonl",'w',encoding="utf-8") as f2:
    with open("D:\github4\web_driver\data_new_handmade-1",'r',encoding="utf-8") as f_hand:
        tok = f_hand.read()
        tok=tok.split('\n--------------new--post--line--------------------------------\n')
        for h in tok:

            h=h.strip()
            t=recursive_chunk(h,split_symbols=[r"\*\*Question:\*\*",r"\*\*Answer:\*\*"])

            t=delete_small_chunks(t,5)


            # assert(len(t)%2==0)
            conversations=[]
            for j in range(int(len(t)/2)):
                conversations+=[{"content":t[j*2].strip(), "role": "user"},{"content": t[j*2+1].strip(), "role": "assistant"}]

            conversations={"messages":conversations}
            f2.write(json.dumps(conversations)+'\n')

In [1]:
import json
as1=[]
with open("D:\github4\web_driver\data2.jsonl",'r',encoding="utf-8") as f2:
    for i in f2:
        as1+=[json.loads(i)]


In [ ]:
i=16
print(as1[i]["conversations"])

In [8]:
assistant_names=["<# Mark_L_Stone #>:","<# jackfsuiaJack #>:","<# ErlingErling D.Andersen #>:","<# echu #>:","<# Stephen_Becker #>:","<# Joachim_Dahl #>:","<# Michal_Adamaszek #>:",\
      "<# hfribergHenrik A. Friberg #>:","<# mcgMichael C. Grant #>:"]


import re

def find_pattern_positions(text, pattern=r'<# .+? #>:'):
    # 编译正则表达式模式
    regex = re.compile(pattern)
    
    # 使用finditer方法找到所有匹配项
    matches = regex.finditer(text)
    
    # 遍历匹配项，输出每个匹配项的起始位置
    positions = [match for match in matches]
    return positions
with open("D:\github4\web_driver\data2.jsonl",'r',encoding="utf-8") as f:
    with open("D:\github4\web_driver\data2-replace-name.jsonl",'w',encoding="utf-8") as f2:
        for i in f:
            q=json.loads(i)
            conv = q['conversations']
            matches=find_pattern_positions(conv)
            name_order=0
            format_conversations=[]

            while name_order<len(matches):

                m=matches[name_order]
                start=m.start()
                end=m.end()
                name=conv[start:end]
                if name not in assistant_names:
                    if name_order < len(matches) -1:
                        pieces = conv[end:matches[name_order+1].start()]
                    else:
                        pieces = conv[end:]
                    pieces = pieces.strip()
                    if format_conversations!=[] and format_conversations[-1]["role"] ==  "user":
                        pieces = format_conversations[-1]["content"] + pieces
                 
                    format_conversations+=[{"content":pieces.strip(), "role": "user"},]#{"content": t[j*2+1].strip(), "role": "assistant"}]
                if name in assistant_names:
                    if name_order < len(matches) -1:
                        pieces = conv[end:matches[name_order+1].start()]
                    else:
                        pieces = conv[end:]
                    pieces = pieces.strip()
                    if format_conversations!=[] and format_conversations[-1]["role"] ==  "assistant":
                        pieces = format_conversations[-1]["content"] + pieces
                 
                    format_conversations+=[{"content":pieces.strip(), "role": "assistant"},]#{"content": t[j*2+1].strip(), "role": "assistant"}]

                name_order+=1

            # if (format_conversations[0]["role"] ==  "assistant"):
            #     format_conversations=[{"content":f"Tell me about {q['title'][:-len('- CVX Forum: a community-driven support forum')]}", "role": "user"},]+format_conversations
 
            conversations={"messages":format_conversations}
            f2.write(json.dumps(conversations)+'\n')
